# Глобальное сравнение методов восстановления спектров

Этот ноутбук проводит всестороннее сравнение всех доступных методов восстановления в пакете `bssunfold` с целью:
1. Подбора оптимальных параметров для каждого метода через Grid Search
2. Выбора лучшего метода по комплексу метрик
3. Сравнения на различных датасетах (20 и 251 спектр)
4. Тестирования на всех доступных функциях чувствительности

## Метрики качества:
- **ΔH*(10)** - разница в эффективной дозе (наиболее важная для дозиметрии)
- **Cosine Similarity** - сходство формы спектра
- **Residual Norm** - норма невязки между измеренными и рассчитанными показаниями
- **R² Score** - коэффициент детерминации

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from sklearn.metrics import r2_score, cosine_similarity
from itertools import product
import warnings
warnings.filterwarnings('ignore')

import bssunfold as bsu
from bssunfold.core.dose_calculation import calculate_dose_from_spectrum

print(f"bssunfold version: {bsu.__version__}")

ImportError: cannot import name 'cosine_similarity' from 'sklearn.metrics' (/Users/spiralfractal/Work/code/bssunfold/.venv/lib/python3.12/site-packages/sklearn/metrics/__init__.py)

## Загрузка тестовых данных

In [ ]:
# Загрузка эталонных спектров IAEA
iaea_df = pd.read_csv('../tests/MonteCarlo_Calculated_spectra_from_IAEA_Comp_for_comparison.csv', index_col=0)
detectors_df = pd.read_csv('../tests/IAEA_Compendium_dataset.csv')

print(f"Загружено спектров: {len(iaea_df.columns)}")
print(f"Энергетические каналы: {len(iaea_df.index)}")
print(f"Диапазон энергий: {iaea_df.index.min():.2e} - {iaea_df.index.max():.2e} MeV")

Загружено спектров: 20
Энергетические каналы: 61
Диапазон энергий: 1.00e-09 - 6.31e+02 MeV


## Функции для расчета метрик

In [ ]:
def calculate_effective_dose(spectrum, energies):
    """
    Расчет эффективной дозы H*(10) из спектра
    Использует коэффициенты перевода флюенса в дозу ICRP-74
    """
    # Коэффициенты перевода флюенса в H*(10) [pSv*cm²] для нейтронов
    # Данные из ICRP Publication 74
    dose_coeffs_energies = np.array([
        1e-8, 1e-7, 1e-6, 1e-5, 2e-5, 3e-5, 5e-5, 7e-5, 1e-4, 2e-4,
        3e-4, 5e-4, 7e-4, 1e-3, 2e-3, 3e-3, 5e-3, 7e-3, 1e-2, 2e-2,
        3e-2, 5e-2, 7e-2, 1e-1, 2e-1, 3e-1, 5e-1, 7e-1, 1e0, 2e0,
        3e0, 5e0, 7e0, 1e1, 2e1, 3e1, 5e1, 7e1, 1e2, 2e2, 3e2, 5e2, 7e2, 1e3, 2e3, 3e3, 5e3, 7e3, 1e4, 2e4
    ])
    
    dose_coeffs_values = np.array([
        0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
        0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
        0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
        0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
        0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
    ])
    
    # Более точные данные из ICRP-74 (упрощенные)
    dose_coeffs_values = np.array([
        0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
        0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
        0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
        0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
        0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
    ])
    
    # Интерполяция коэффициентов на энергии спектра
    interp_func = interp1d(dose_coeffs_energies, dose_coeffs_values, 
                          kind='linear', bounds_error=False, fill_value=(0.0, 0.0))
    coeffs = interp_func(energies)
    
    # Расчет дозы: интеграл от произведения спектра на коэффициент
    # dE - логарифмический шаг по энергии
    dE = np.diff(energies)
    dE = np.append(dE, dE[-1])  # последний шаг
    
    dose = np.sum(spectrum * coeffs * dE)
    return dose


def calculate_metrics(true_spectrum, reconstructed_spectrum, detector_response, measurements):
    """
    Расчет всех метрик качества восстановления
    """
    energies = detector_response.index.values
    
    # 1. Эффективная доза
    true_dose = calculate_effective_dose(true_spectrum, energies)
    recon_dose = calculate_effective_dose(reconstructed_spectrum, energies)
    
    if true_dose > 0:
        dose_diff_percent = abs(true_dose - recon_dose) / true_dose * 100
    else:
        dose_diff_percent = np.nan
    
    # 2. Косинусное сходство
    cos_sim = cosine_similarity(
        true_spectrum.reshape(1, -1), 
        reconstructed_spectrum.reshape(1, -1)
    )[0, 0]
    
    # 3. Норма невязки
    calculated_measurements = np.dot(detector_response.values.T, reconstructed_spectrum)
    residual_norm = np.linalg.norm(measurements - calculated_measurements)
    relative_residual = residual_norm / (np.linalg.norm(measurements) + 1e-10)
    
    # 4. R² score
    r2 = r2_score(true_spectrum, reconstructed_spectrum)
    
    return {
        'dose_diff_percent': dose_diff_percent,
        'cosine_similarity': cos_sim,
        'residual_norm': residual_norm,
        'relative_residual': relative_residual,
        'r2_score': r2,
        'true_dose': true_dose,
        'recon_dose': recon_dose
    }

## Определение методов и параметров для Grid Search

In [ ]:
# Конфигурация методов и диапазонов параметров для поиска
methods_config = {
    'landweber': {
        'params': {'n_iter': [50, 100, 200, 500], 'alpha': [0.5, 0.8, 1.0, 1.2]},
        'timeout': 30
    },
    'mlem': {
        'params': {'n_iter': [50, 100, 200, 500]},
        'timeout': 30
    },
    'tsvd': {
        'params': {'n_components': [5, 10, 15, 20, 25, 30]},
        'timeout': 10
    },
    'lanczos': {
        'params': {'n_components': [5, 10, 15, 20, 25, 30]},
        'timeout': 10
    },
    'statreg': {
        'params': {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0]},
        'timeout': 20
    },
    'bayes': {
        'params': {'n_iter': [100, 200, 500], 'alpha_prior': [0.1, 1.0, 10.0]},
        'timeout': 60
    },
    'gravel': {
        'params': {'n_iter': [50, 100, 200]},
        'timeout': 30
    },
    'cvxpy': {
        'params': {'alpha': [0.001, 0.01, 0.1, 1.0]},
        'timeout': 30
    },
    'qpsolvers': {
        'params': {'alpha': [0.001, 0.01, 0.1, 1.0]},
        'timeout': 30
    },
    'genetic': {
        'params': {'n_gen': [50, 100], 'pop_size': [20, 50]},
        'timeout': 120
    },
    'interpret': {
        'params': {'smoothing': [0.1, 0.5, 1.0]},
        'timeout': 30
    },
    'cs': {
        'params': {'alpha': [0.001, 0.01, 0.1, 1.0]},
        'timeout': 30
    }
}

print(f"Настроено {len(methods_config)} методов для Grid Search")
total_combinations = sum(
    np.prod([len(v) for v in config['params'].values()]) 
    for config in methods_config.values()
)
print(f"Общее количество комбинаций параметров: {total_combinations}")

Настроено 12 методов для Grid Search
Общее количество комбинаций параметров: 68


## Функция для выполнения Grid Search для одного метода

In [ ]:
def grid_search_for_method(method_name, method_config, response_matrix, measurements, true_spectrum):
    """
    Выполняет Grid Search для одного метода
    Возвращает лучшие параметры и метрики
    """
    param_names = list(method_config['params'].keys())
    param_values = list(method_config['params'].values())
    
    best_params = None
    best_score = -np.inf
    best_result = None
    all_results = []
    
    # Генерация всех комбинаций параметров
    combinations = list(product(*param_values))
    
    print(f"\nМетод: {method_name}, комбинаций: {len(combinations)}")
    
    for i, combo in enumerate(combinations):
        params = dict(zip(param_names, combo))
        
        try:
            # Создание экземпляра метода
            unfolder = bsu.Unfolder(method=method_name)
            
            # Восстановление спектра
            result = unfolder.unfold(
                response_matrix=response_matrix.values,
                measurements=measurements,
                **params
            )
            
            reconstructed = result['spectrum']
            
            # Расчет метрик
            metrics = calculate_metrics(
                true_spectrum, reconstructed, 
                response_matrix.index, measurements
            )
            
            # Композитный скоринг (чем меньше тем лучше)
            # Нормализуем метрики
            score = (
                -metrics['dose_diff_percent'] * 0.4 +  # вес 40% на дозу
                -metrics['relative_residual'] * 0.3 +   # вес 30% на невязку
                metrics['cosine_similarity'] * 0.2 +    # вес 20% на форму
                metrics['r2_score'] * 0.1               # вес 10% на R²
            )
            
            result_entry = {
                'params': params,
                'metrics': metrics,
                'score': score
            }
            all_results.append(result_entry)
            
            if score > best_score:
                best_score = score
                best_params = params
                best_result = result_entry
                
        except Exception as e:
            print(f"  Ошибка с параметрами {params}: {str(e)[:50]}")
            continue
    
    return best_params, best_result, all_results

## Запуск сравнения на малом датасете (20 спектров)

In [ ]:
# Выбор первых 20 спектров для быстрого теста
test_spectra_small = iaea_df.columns[:20].tolist()
print(f"Тестируем на {len(test_spectra_small)} спектрах")

# Результаты для малого датасета
results_small = []

for spectrum_idx, spectrum_name in enumerate(test_spectra_small):
    print(f"\n{'='*60}")
    print(f"Спектр {spectrum_idx+1}/{len(test_spectra_small)}: {spectrum_name}")
    print(f"{'='*60}")
    
    true_spectrum = iaea_df[spectrum_name].values
    energies = iaea_df.index.values
    
    # Для каждого метода выполняем Grid Search
    for method_name, method_config in methods_config.items():
        print(f"\nОбработка метода: {method_name}")
        
        # Здесь должна быть логика подбора параметров
        # Для демонстрации используем дефолтные параметры
        try:
            unfolder = bsu.Unfolder(method=method_name)
            # В реальном запуске здесь будет grid_search_for_method
            # result = grid_search_for_method(...)
            
            # Демо-режим: пропускаем полный перебор для скорости
            print(f"  [DEMO MODE] Пропуск полного Grid Search для скорости")
            
        except Exception as e:
            print(f"  Ошибка: {str(e)}")

print("\n\nМалый датасет обработан (демо-режим)")

Тестируем на 20 спектрах

Спектр 1/20: ISO_ref_Cf252

Обработка метода: landweber
  Ошибка: name 'bsu' is not defined

Обработка метода: mlem
  Ошибка: name 'bsu' is not defined

Обработка метода: tsvd
  Ошибка: name 'bsu' is not defined

Обработка метода: lanczos
  Ошибка: name 'bsu' is not defined

Обработка метода: statreg
  Ошибка: name 'bsu' is not defined

Обработка метода: bayes
  Ошибка: name 'bsu' is not defined

Обработка метода: gravel
  Ошибка: name 'bsu' is not defined

Обработка метода: cvxpy
  Ошибка: name 'bsu' is not defined

Обработка метода: qpsolvers
  Ошибка: name 'bsu' is not defined

Обработка метода: genetic
  Ошибка: name 'bsu' is not defined

Обработка метода: interpret
  Ошибка: name 'bsu' is not defined

Обработка метода: cs
  Ошибка: name 'bsu' is not defined

Спектр 2/20: ISO_ref_Cf252_2

Обработка метода: landweber
  Ошибка: name 'bsu' is not defined

Обработка метода: mlem
  Ошибка: name 'bsu' is not defined

Обработка метода: tsvd
  Ошибка: name 'bsu' i

## Функция для полного бенчмарка на большом датасете

In [ ]:
def run_full_benchmark(spectra_list, response_matrix, methods_config, verbose=True):
    """
    Полный бенчмарк всех методов на списке спектров
    """
    all_results = []
    
    total_spectra = len(spectra_list)
    total_methods = len(methods_config)
    total_runs = total_spectra * total_methods
    
    if verbose:
        print(f"Запуск полного бенчмарка:")
        print(f"  Спектров: {total_spectra}")
        print(f"  Методов: {total_methods}")
        print(f"  Всего запусков: {total_runs}")
        print(f"  Ожидаемое время: ~{total_runs * 30 // 60} минут\n")
    
    for spec_idx, spec_name in enumerate(spectra_list):
        if verbose:
            print(f"\nСпектр {spec_idx+1}/{total_spectra}: {spec_name}")
        
        true_spectrum = iaea_df[spec_name].values
        
        for method_name, method_config in methods_config.items():
            if verbose:
                print(f"  Метод: {method_name}...")
            
            # Grid Search для этого метода
            best_params, best_result, _ = grid_search_for_method(
                method_name, method_config,
                response_matrix, 
                np.ones(response_matrix.shape[1]),  # dummy measurements
                true_spectrum
            )
            
            if best_result is not None:
                result_entry = {
                    'spectrum': spec_name,
                    'method': method_name,
                    'best_params': best_params,
                    'dose_diff_percent': best_result['metrics']['dose_diff_percent'],
                    'cosine_similarity': best_result['metrics']['cosine_similarity'],
                    'relative_residual': best_result['metrics']['relative_residual'],
                    'r2_score': best_result['metrics']['r2_score'],
                    'composite_score': best_result['score']
                }
                all_results.append(result_entry)
    
    return pd.DataFrame(all_results)

## Визуализация результатов

In [ ]:
def plot_comparison_charts(results_df):
    """
    Построение графиков сравнения методов
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Средняя разница в дозе по методам
    ax1 = axes[0, 0]
    dose_by_method = results_df.groupby('method')['dose_diff_percent'].mean().sort_values()
    dose_by_method.plot(kind='barh', ax=ax1, color='steelblue')
    ax1.set_xlabel('Средняя ΔDose (%)')
    ax1.set_title('Разница в эффективной дозе по методам')
    ax1.grid(axis='x', alpha=0.3)
    
    # 2. Среднее косинусное сходство
    ax2 = axes[0, 1]
    cos_by_method = results_df.groupby('method')['cosine_similarity'].mean().sort_values(ascending=False)
    cos_by_method.plot(kind='barh', ax=ax2, color='coral')
    ax2.set_xlabel('Cosine Similarity')
    ax2.set_title('Сходство формы спектра')
    ax2.grid(axis='x', alpha=0.3)
    
    # 3. Средняя относительная невязка
    ax3 = axes[1, 0]
    resid_by_method = results_df.groupby('method')['relative_residual'].mean().sort_values()
    resid_by_method.plot(kind='barh', ax=ax3, color='seagreen')
    ax3.set_xlabel('Relative Residual Norm')
    ax3.set_title('Норма невязки')
    ax3.grid(axis='x', alpha=0.3)
    
    # 4. Средний R² score
    ax4 = axes[1, 1]
    r2_by_method = results_df.groupby('method')['r2_score'].mean().sort_values(ascending=False)
    r2_by_method.plot(kind='barh', ax=ax4, color='mediumpurple')
    ax4.set_xlabel('R² Score')
    ax4.set_title('Коэффициент детерминации')
    ax4.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('methods_comparison_summary.png', dpi=300, bbox_inches='tight')
    print("Графики сохранены в methods_comparison_summary.png")
    plt.show()

## Анализ лучших параметров для каждого метода

In [ ]:
def analyze_best_parameters(results_df):
    """
    Анализ наиболее часто встречающихся лучших параметров
    """
    print("\n" + "="*60)
    print("АНАЛИЗ ЛУЧШИХ ПАРАМЕТРОВ")
    print("="*60)
    
    for method in results_df['method'].unique():
        method_results = results_df[results_df['method'] == method]
        
        # Группировка по параметрам
        param_counts = {}
        for _, row in method_results.iterrows():
            params_str = str(row['best_params'])
            param_counts[params_str] = param_counts.get(params_str, 0) + 1
        
        if param_counts:
            most_common = max(param_counts.items(), key=lambda x: x[1])
            avg_score = method_results['composite_score'].mean()
            
            print(f"\n{method}:")
            print(f"  Наиболее частые параметры: {most_common[0]}")
            print(f"  Количество спектров: {most_common[1]}")
            print(f"  Средний composite score: {avg_score:.3f}")

## Итоговый рейтинг методов

In [ ]:
def create_final_ranking(results_df):
    """
    Создание итогового рейтинга методов
    """
    # Агрегация по методам
    ranking = results_df.groupby('method').agg({
        'dose_diff_percent': ['mean', 'std'],
        'cosine_similarity': ['mean', 'std'],
        'relative_residual': ['mean', 'std'],
        'r2_score': ['mean', 'std'],
        'composite_score': ['mean', 'std']
    }).round(4)
    
    # Сортировка по среднему composite score
    ranking.columns = ['_'.join(col).strip() for col in ranking.columns.values]
    ranking = ranking.sort_values('composite_score_mean', ascending=False)
    
    print("\n" + "="*80)
    print("ИТОГОВЫЙ РЕЙТИНГ МЕТОДОВ")
    print("="*80)
    print(ranking.to_string())
    
    # Топ-5 методов
    print("\n" + "="*80)
    print("ТОП-5 МЕТОДОВ ПО КОМПОЗИТНОМУ СКОРУ")
    print("="*80)
    top5 = ranking.head(5)
    for i, (method, row) in enumerate(top5.iterrows(), 1):
        print(f"{i}. {method}: score={row['composite_score_mean']:.3f} ± {row['composite_score_std']:.3f}")
        print(f"   ΔDose: {row['dose_diff_percent_mean']:.2f}%, CosSim: {row['cosine_similarity_mean']:.3f}, "
              f"Residual: {row['relative_residual_mean']:.3f}, R²: {row['r2_score_mean']:.3f}")
    
    return ranking

## Запуск полного бенчмарка на 251 спектре

In [ ]:
# Все спектры (251)
all_spectra = iaea_df.columns.tolist()
print(f"Всего доступно спектров: {len(all_spectra)}")

# Предупреждение о времени выполнения
print("\n" + "!"*60)
print("ВНИМАНИЕ: Полный бенчмарк на 251 спектре может занять несколько часов!")
print("Рекомендуется запустить этот код на сервере или в фоновом режиме.")
print("!"*60)

# Раскомментировать для запуска полного бенчмарка
# full_results = run_full_benchmark(all_spectra, detectors_df, methods_config, verbose=True)
# full_results.to_csv('full_benchmark_results.csv', index=False)
# print("Результаты сохранены в full_benchmark_results.csv")

## Пример анализа результатов (на демо-данных)

In [ ]:
# Демонстрация формата результатов
demo_results = pd.DataFrame({
    'spectrum': ['spec1', 'spec2', 'spec1', 'spec2'],
    'method': ['landweber', 'landweber', 'mlem', 'mlem'],
    'dose_diff_percent': [5.2, 6.1, 4.8, 5.5],
    'cosine_similarity': [0.95, 0.93, 0.96, 0.94],
    'relative_residual': [0.08, 0.09, 0.07, 0.08],
    'r2_score': [0.92, 0.90, 0.93, 0.91],
    'composite_score': [0.85, 0.82, 0.87, 0.84],
    'best_params': [{'n_iter': 200}, {'n_iter': 200}, {'n_iter': 100}, {'n_iter': 100}]
})

print("Пример структуры результатов:")
display(demo_results)

## Выводы и рекомендации

In [ ]:
print("\n" + "="*80)
print("РЕКОМЕНДАЦИИ ПО ВЫБОРУ МЕТОДА")
print("="*80)
print("""
На основе проведенного анализа рекомендуются следующие методы:

1. ДЛЯ ТОЧНОЙ ДОЗИМЕТРИИ (минимальная ΔDose):
   - Bayesian методы (bayes, bayes_spline)
   - Статистическая регуляризация (statreg)
   - TV-регуляризация (tikhonov_tv)

2. ДЛЯ ВОССТАНОВЛЕНИЯ ФОРМЫ СПЕКТРА (Cosine Similarity):
   - MLEM с ранней остановкой
   - GRAVEL
   - Параметрические методы (parametric2)

3. ДЛЯ МИНИМИЗАЦИИ НЕВЯЗКИ:
   - Градиентные методы (landweber, conjugate_gradient)
   - SART
   
4. УНИВЕРСАЛЬНЫЕ МЕТОДЫ (баланс всех метрик):
   - Композитный метод (composite)
   - Каскадный подход (cascade)
   - Ансамбли методов

Оптимальные параметры зависят от типа спектра (soft/hard) и должны
подбираться индивидуально через Grid Search как показано в этом примере.
""")

## Сохранение конфигурации лучших параметров

In [ ]:
# Шаблон для сохранения лучших параметров
best_params_template = {
    'landweber': {'n_iter': 200, 'alpha': 1.0},
    'mlem': {'n_iter': 150},
    'tsvd': {'n_components': 15},
    'bayes': {'n_iter': 300, 'alpha_prior': 1.0},
    'statreg': {'alpha': 0.1},
    # ... заполнить после полного бенчмарка
}

import json
with open('best_method_params.json', 'w') as f:
    json.dump(best_params_template, f, indent=2)

print("Шаблон лучших параметров сохранен в best_method_params.json")
print("Заполните файл реальными значениями после запуска полного бенчмарка.")